In [12]:
# import needed libraries
import os
import numpy as np
import open3d as o3d

# import custom libraries
import al_utils as al

print("Open3D version:", o3d.__version__)

Open3D version: 0.18.0


In [13]:
# Configuration
USED_PCD_ID = 300
USED_PCD_FILE_DIR = 'used_pcd'
SRC_PCD_FILE_DIR = 'points'

In [14]:
# read pcd file
pcd = o3d.io.read_point_cloud(al.get_pcd_file_path(SRC_PCD_FILE_DIR, USED_PCD_ID, USED_PCD_FILE_DIR))

# print pcd info
total_points = len(pcd.points)

# remove nan points and infinite points
pcd = pcd.remove_non_finite_points(remove_nan=True, remove_infinite=True)

# print pcd info
print(f"Valid points: {len(pcd.points)}, valid ratio: {len(pcd.points) / total_points * 100:.2f}%")

# Visualize if needed
# al.show_point_cloud(pcd, name=f"Original Point Cloud: {len(pcd.points)} points")

al.show_geometries([pcd, al.create_coordinate(1.5)], name=f"Original Point Cloud: {len(pcd.points)} points")


Working directory: /root/AC1/autopath
Path is valid
Total 741 pcd files found, total size: 475.11 MB
Target pcd file: 1771498195.700643062.pcd, size: 658.27 KB
Using No.300 pcd file: /root/AC1/autopath/points/1771498195.700643062.pcd, basename: 1771498195.700643062.pcd
Copy pcd file to used_pcd folder...
File 1771498195.700643062.pcd already exists in used_pcd folder, Checking if it is the same file...
File 1771498195.700643062.pcd is the same file, no need to copy
Valid points: 17628, valid ratio: 63.76%


In [15]:
# Downsample
downpcd = pcd.voxel_down_sample(voxel_size=0.1)
# o3d.visualization.draw_geometries([downpcd])
# print downpcd info
print(f"Downsampled points: {len(downpcd.points)}, downsample ratio: {len(downpcd.points) / len(pcd.points) * 100:.2f}%", )

# show_point_cloud(downpcd, name=f"Downsampled Point Cloud: {len(downpcd.points)} points")

# Remove outliers
cl, ind = downpcd.remove_statistical_outlier(nb_neighbors=10, std_ratio=1.0)
filtered_pcd = downpcd.select_by_index(ind)
print(f"Filtered points: {len(filtered_pcd.points)}, Filtered ratio: {len(filtered_pcd.points) / len(downpcd.points) * 100:.2f}%")

# Visualize if needed
al.show_geometries([al.add_origin(filtered_pcd), al.create_coordinate(1.5)], name=f"Filtered Point Cloud: {len(filtered_pcd.points)} points")
# al.show_point_cloud(al.add_origin(filtered_pcd), name=f"Filtered Point Cloud: {len(filtered_pcd.points)} points")


Downsampled points: 8180, downsample ratio: 46.40%
Filtered points: 7185, Filtered ratio: 87.84%


In [16]:
all_geometries = [al.add_origin(filtered_pcd), al.create_line_set([[0, 0, 0], [0, 0, 1], [0, 1, 0], [1, 0, 0]]), al.create_coordinate(1.5)]
al.show_geometries(all_geometries)


In [17]:
planes = al.find_planes(filtered_pcd, distance_threshold=0.05, ransac_n=3, num_iterations=1000)

print(f"Found {len(planes)} planes")

Found 51 planes


In [18]:
# show all planes
color_list = [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0], [1, 0, 1], [0, 1, 1], [1, 0.5, 0], [0.5, 1, 0], [0.5, 0, 1],[0.5, 0.5, 1],[1, 0.5, 0.5], [0.5, 1, 0.5], [0.5, 0.5, 1]]
planes_pcd = o3d.geometry.PointCloud()
minimum_points = 50
plane_cnt = 0
for i in range(len(planes)):
    plane_model, plane_pcd = planes[i]

    # only keep the plane horizontal
    [a, b, c, d] = plane_model
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)
    z_component = abs(normal[2])
    if z_component < 0.9:
        continue

    # skip small planes
    if len(plane_pcd.points) < minimum_points:
        continue

    # show_point_cloud(plane_pcd, name=f"Plane Point Cloud: {len(plane_pcd.points)} points")
    al.add_color(plane_pcd, color=color_list[i % len(color_list)])
    planes_pcd = al.merge_pcds([planes_pcd, plane_pcd])
    plane_cnt += 1

al.show_point_cloud(al.add_origin(planes_pcd), name=f"Total {plane_cnt} planes: {len(planes_pcd.points)} points")

# calculate the centroid of the planes
centroid = planes_pcd.get_center()
print(f"Centroid of the planes: {centroid}")
    

Centroid of the planes: [ 3.90707645 -0.42041957  0.04084741]


In [19]:
import matplotlib.pyplot as plt

def detect_horizontal_plane(pcd, distance_threshold=0.05, ransac_n=3, num_iterations=1000):
    """
    从点云中检测水平面（法向量接近Z轴）
    :param pcd: 预处理后的点云对象
    :param distance_threshold: 点到平面的距离阈值
    :param ransac_n: RANSAC每次采样的点数
    :param num_iterations: RANSAC迭代次数
    :return: 平面参数（ax+by+cz+d=0）、平面点云、平面中心坐标
    """
    # 使用RANSAC拟合平面
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations
    )
    [a, b, c, d] = plane_model
    print(f"拟合平面方程: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")
    
    # 筛选水平面（法向量z分量绝对值接近1，即法向量接近Z轴）
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)  # 归一化法向量
    z_component = abs(normal[2])
    if z_component < 0.9:  # 法向量z分量小于0.9，认为不是水平面
        raise ValueError(f"检测到的平面不是水平面（法向量z分量: {z_component:.4f}）")
    
    # 提取平面内的点云
    plane_pcd = pcd.select_by_index(inliers)
    plane_pcd = al.add_color(plane_pcd, [1, 1, 0])
    # plane_pcd.paint_uniform_color([1, 0, 0])  # 红色标记水平面
    
    # 计算平面中心坐标
    plane_points = np.asarray(plane_pcd.points)
    plane_center = np.mean(plane_points, axis=0)
    print(f"Center of plane: {plane_center}")
    
    return plane_model, plane_pcd, plane_center

def plan_landing_path(plane_center, safe_height=2.0, step_num=50):
    """
    规划无人机降落路径：原点 -> 目标点正上方-> 垂直降落至平面中心
    :param plane_center: 水平面中心坐标 [x, y, z]
    :param safe_height: 目标点正上方的安全高度（相对于平面）
    :param step_num: 路径点数量（均分）
    :return: 降落路径点云、路径点坐标数组
    """
    # 路径关键点
    start_point = np.array([0, 0, 0])  # 起点（原点）
    hover_point = np.array([plane_center[0], plane_center[1], plane_center[2] + safe_height])  # 悬停点（安全高度）
    land_point = plane_center  # 降落点（平面中心）
    
    # Generate segmented path
    # 第一段：原点到悬停点（直线）
    path_1 = np.linspace(start_point, hover_point, step_num // 2)
    # 第二段：悬停点到降落点（垂直下降）
    path_2 = np.linspace(hover_point, land_point, step_num // 2)
    # 合并路径
    path_points = np.vstack((path_1, path_2))
    
    # 创建路径点云（蓝色标记）
    path_pcd = o3d.geometry.PointCloud()
    path_pcd.points = o3d.utility.Vector3dVector(path_points)
    path_pcd = al.add_color(path_pcd, [0, 0, 1])
    
    print(f"降落路径规划完成：")
    print(f"  - 起点: {start_point}")
    print(f"  - 悬停点: {hover_point} (安全高度: {safe_height}m)")
    print(f"  - 降落点: {land_point}")
    
    return path_pcd, path_points


In [20]:
plane_model, plane_pcd, plane_center = detect_horizontal_plane(filtered_pcd)

拟合平面方程: -0.1446x + 0.1025y + 0.9842z + 1.8426 = 0
Center of plane: [ 4.0108172  -1.00804955 -1.17781028]


In [21]:
path_pcd, path_points = plan_landing_path(plane_center, safe_height=1.5)

降落路径规划完成：
  - 起点: [0 0 0]
  - 悬停点: [ 4.0108172  -1.00804955  0.32218972] (安全高度: 1.5m)
  - 降落点: [ 4.0108172  -1.00804955 -1.17781028]


In [22]:
al.show_geometries([al.create_coordinate(1.5), al.add_origin(filtered_pcd), plane_pcd, al.smooth_pcd_path(path_pcd, 9)], name="Path Planning to Biggest Plane Center")